# MuSeg — Asian Dataset Evaluation (Thigh, Water-only & Dixon-based)

Computes per-muscle Dice / Hausdorff / etc. for MuSeg segmentations on the
**MRI_data_asian** thigh dataset against the unilateral ground truth.

Two variants are evaluated in separate sections:

| Variant | Seg dir | Result dir |
|---|---|---|
| Water-only | `../asian_segs/water_only/` | `results_asian_water_only/` |
| Dixon-based | `../asian_segs/dixon_based/` | `results_asian_dixon_based/` |

**MuSeg label map** (no L/R distinction — both sides share one label):

| Muscle | Asian GT label | MuSeg label(s) |
|---|---|---|
| rectus_femoris | 1 | 4 |
| vastus_lateralis | 2 | 1 |
| vastus_intermedius | 3 | 2 |
| vastus_medialis | 4 | 3 |
| sartorius | 5 | 5 |
| gracilis | 6 | 6 |
| biceps_femoris | 7 | 9 + 10 (long + short merged) |
| semitendinosus | 8 | 8 |
| semimembranosus | 9 | 7 |
| adductor_brevis | 10 | 13 |
| adductor_longus | 11 | 12 |
| adductor_magnus | 12 | 11 |
| gluteus_maximus | 13 | — (not in MuSeg) |

**Note:** The Asian GT is unilateral — each label covers only one side.  
MuSeg predicts both sides under the same label, so false positives on the
unlabelled side are expected. Results should be interpreted accordingly.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
SEG_BASE   = os.path.join('..', 'asian_segs')

SEG_DIR_WATER_ONLY  = os.path.join(SEG_BASE, 'water_only')
SEG_DIR_DIXON_BASED = os.path.join(SEG_BASE, 'dixon_based')

RESULT_DIR_WATER_ONLY  = 'results_asian_water_only'
RESULT_DIR_DIXON_BASED = 'results_asian_dixon_based'

os.makedirs(RESULT_DIR_WATER_ONLY,  exist_ok=True)
os.makedirs(RESULT_DIR_DIXON_BASED, exist_ok=True)

# (muscle_name, asian_gt_label, museg_labels)
# museg_labels: list of MuSeg integer labels to OR together, or None if not in model.
# No L/R split — both sides predicted under same label vs unilateral GT.
MUSCLES = [
    ('rectus_femoris',     1,  [4]),       # MuSeg: Rectus_Femoris
    ('vastus_lateralis',   2,  [1]),       # MuSeg: Vastus_Lateralis
    ('vastus_intermedius', 3,  [2]),       # MuSeg: Vastus_Intermedius
    ('vastus_medialis',    4,  [3]),       # MuSeg: Vastus_Medialis
    ('sartorius',          5,  [5]),       # MuSeg: Sartorius
    ('gracilis',           6,  [6]),       # MuSeg: Gracilis
    ('biceps_femoris',     7,  [9, 10]),   # MuSeg: Biceps_Femoris + Biceps_Femoris_Short
    ('semitendinosus',     8,  [8]),       # MuSeg: Semitendinosus
    ('semimembranosus',    9,  [7]),       # MuSeg: Semimembranosus
    ('adductor_brevis',    10, [13]),      # MuSeg: Adductor_Brevis
    ('adductor_longus',    11, [12]),      # MuSeg: Adductor_Longus
    ('adductor_magnus',    12, [11]),      # MuSeg: Adductor_Magnus
    ('gluteus_maximus',    13, None),      # Not in MuSeg → all-zero prediction
]

for label, path in [
    ('DATA_ROOT',    DATA_ROOT),
    ('Water-only',   SEG_DIR_WATER_ONLY),
    ('Dixon-based',  SEG_DIR_DIXON_BASED),
]:
    print(f'{label}: {os.path.abspath(path)}  exists={os.path.isdir(path)}')

In [ ]:
def evaluate_muscle(muscle_name, gt_label, museg_labels, seg_dir, result_dir, csv_suffix):
    """
    museg_labels: list of integer labels to OR, or None (→ all-zero pred).
    Segmentation files expected at: seg_dir/{subject}/Thigh/Thigh_seg.nii.gz
    """
    seg_files = sorted(glob.glob(os.path.join(seg_dir, '*', 'Thigh', 'Thigh_seg.nii.gz')))
    if not seg_files:
        print(f'  [skip] no segmentation files in {seg_dir}')
        return pd.DataFrame()

    results = []
    for seg_path in seg_files:
        parts   = seg_path.replace('\\', '/').split('/')
        subject = parts[-3]

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_raw = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
        if museg_labels is not None:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)
            for lbl in museg_labels:
                pred_arr |= (seg_raw == lbl).astype(np.uint8)
        else:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(seg_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_museg_{csv_suffix}.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df


print('evaluate_muscle ready.')

## Water-only

In [ ]:
dfs_water = {}
for muscle_name, gt_label, museg_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, museg={museg_labels}) ──')
    dfs_water[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, museg_labels,
        SEG_DIR_WATER_ONLY, RESULT_DIR_WATER_ONLY, 'asian_water_only',
    )
print('\nDone.')

In [ ]:
# ── Water-only summary ────────────────────────────────────────────────────────
from IPython.display import display

summary_rows = []
for muscle_name, df in dfs_water.items():
    if df.empty:
        continue
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary_water = pd.DataFrame(summary_rows).set_index('muscle')
summary_path  = os.path.join(RESULT_DIR_WATER_ONLY, 'summary_museg_asian_water_only.csv')
summary_water.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary_water.round(4))

## Dixon-based

In [ ]:
dfs_dixon = {}
for muscle_name, gt_label, museg_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, museg={museg_labels}) ──')
    dfs_dixon[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, museg_labels,
        SEG_DIR_DIXON_BASED, RESULT_DIR_DIXON_BASED, 'asian_dixon_based',
    )
print('\nDone.')

In [ ]:
# ── Dixon-based summary ───────────────────────────────────────────────────────
summary_rows = []
for muscle_name, df in dfs_dixon.items():
    if df.empty:
        continue
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary_dixon = pd.DataFrame(summary_rows).set_index('muscle')
summary_path  = os.path.join(RESULT_DIR_DIXON_BASED, 'summary_museg_asian_dixon_based.csv')
summary_dixon.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary_dixon.round(4))

## Side-by-side comparison

In [ ]:
if not summary_water.empty and not summary_dixon.empty:
    compare = pd.concat([
        summary_water[['dice_mean', 'hausdorff_mean']].rename(
            columns={'dice_mean': 'dice_water', 'hausdorff_mean': 'hd_water'}),
        summary_dixon[['dice_mean', 'hausdorff_mean']].rename(
            columns={'dice_mean': 'dice_dixon', 'hausdorff_mean': 'hd_dixon'}),
    ], axis=1)
    compare['dice_delta'] = compare['dice_dixon'] - compare['dice_water']
    display(compare.round(4))
    compare.to_csv('comparison_water_vs_dixon.csv', float_format='%.4f')
    print('\nSaved comparison_water_vs_dixon.csv')